# 09p — Person ReID: TransReID ViT-B/16 CLIP on Market-1501
Train a person re-identification model using TransReID ViT-Base/16 with CLIP pre-training on the Market-1501 dataset.

**Architecture**: TransReID ViT-Base/16 (CLIP init)
**Dataset**: Market-1501 (751 train IDs, 6 cameras)
**Input**: 256×128 (standard person aspect)
**Expected**: mAP ~89-91%, Rank-1 ~95-96%

In [ ]:
import subprocess, sys

# Install timm for ViT backbone
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9"])

import os, json, random, math, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from PIL import Image

print(f"PyTorch {torch.__version__}, CUDA {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# ── Config ──────────────────────────────────────────────────
CFG = {
    "dataset_name": "market1501",
    "img_size": (256, 128),          # H×W — standard person aspect ratio
    "batch_size": 64,                # P=16 IDs × K=4 instances
    "P": 16,                         # IDs per batch
    "K": 4,                          # instances per ID
    "epochs": 120,
    "lr": 3.5e-4,
    "weight_decay": 5e-4,
    "warmup_epochs": 10,
    "label_smooth": 0.1,
    "triplet_margin": 0.3,
    "embed_dim": 768,
    "num_cameras": 6,                # Market-1501 has 6 cameras
    "eval_every": 10,
    "fp16": True,
}

# Market-1501 dataset — find it from Kaggle datasets
MARKET_ROOTS = [
    "/kaggle/input/market-1501",
    "/kaggle/input/market1501",
    "/kaggle/input/market-1501-dataset",
    "/kaggle/input/datasets/mrkdagods/market-1501",
]
MARKET_ROOT = None
for root in MARKET_ROOTS:
    for sub in [root, os.path.join(root, "Market-1501-v15.09.15"), os.path.join(root, "Market-1501")]:
        if os.path.isdir(os.path.join(sub, "bounding_box_train")):
            MARKET_ROOT = sub
            break
    if MARKET_ROOT:
        break

if not MARKET_ROOT:
    # Try finding it dynamically
    import glob
    candidates = glob.glob("/kaggle/input/**/bounding_box_train", recursive=True)
    if candidates:
        MARKET_ROOT = str(Path(candidates[0]).parent)

assert MARKET_ROOT is not None, f"Market-1501 not found! Searched: {MARKET_ROOTS}"
print(f"Market-1501 root: {MARKET_ROOT}")

# CLIP pretrained ViT weights
PRETRAINED_PATH = None
for p in [
    "/kaggle/input/datasets/mrkdagods/mtmc-weights/reid/ViT-B-16.pt",
    "/kaggle/input/mtmc-weights/reid/ViT-B-16.pt",
]:
    if os.path.isfile(p):
        PRETRAINED_PATH = p
        break
print(f"CLIP ViT weights: {PRETRAINED_PATH}")

In [ ]:
# ── Parse Market-1501 ─────────────────────────────────────
def parse_market1501(root):
    """Parse Market-1501 directory structure.
    Filename format: XXXX_cYsZ_NNNNNN_NN.jpg
    XXXX = person ID, Y = camera (1-6), Z = sequence
    IDs: 0000 = junk, -1 = distractor
    """
    splits = {}
    for split_name, folder in [("train", "bounding_box_train"), ("query", "query"), ("gallery", "bounding_box_test")]:
        split_dir = os.path.join(root, folder)
        samples = []
        for fname in sorted(os.listdir(split_dir)):
            if not fname.endswith((".jpg", ".png")):
                continue
            pid_str = fname.split("_")[0]
            pid = int(pid_str)
            if pid < 0:  # distractor
                continue
            if pid == 0 and split_name == "train":  # junk in train
                continue
            # Camera: c1s1 -> cam 0
            cam_str = fname.split("_")[1]
            cam_id = int(cam_str[1]) - 1  # 0-indexed
            samples.append({
                "path": os.path.join(split_dir, fname),
                "pid": pid,
                "cam_id": cam_id,
            })
        splits[split_name] = samples

    # Remap PIDs to contiguous 0..N-1
    train_pids = sorted(set(s["pid"] for s in splits["train"]))
    pid_map = {pid: idx for idx, pid in enumerate(train_pids)}
    for s in splits["train"]:
        s["label"] = pid_map[s["pid"]]

    n_train_ids = len(train_pids)
    n_train_imgs = len(splits["train"])
    n_query = len(splits["query"])
    n_gallery = len(splits["gallery"])
    print(f"Market-1501: {n_train_ids} train IDs, {n_train_imgs} train images, "
          f"{n_query} query, {n_gallery} gallery")
    return splits, n_train_ids

splits, num_classes = parse_market1501(MARKET_ROOT)
CFG["num_classes"] = num_classes

In [ ]:
# ── Dataset & Sampler ───────────────────────────────────────
class ReIDDataset(Dataset):
    def __init__(self, samples, transform=None, is_train=True):
        self.samples = samples
        self.transform = transform
        self.is_train = is_train

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.is_train:
            return img, s["label"], s["cam_id"]
        return img, s["pid"], s["cam_id"]


class PKSampler(Sampler):
    """Sample P identities, K instances each."""
    def __init__(self, samples, P, K):
        self.P, self.K = P, K
        self.pid_to_indices = defaultdict(list)
        for i, s in enumerate(samples):
            self.pid_to_indices[s["label"]].append(i)
        self.pids = list(self.pid_to_indices.keys())

    def __iter__(self):
        random.shuffle(self.pids)
        batch = []
        for pid in self.pids:
            idxs = self.pid_to_indices[pid]
            if len(idxs) < self.K:
                sel = random.choices(idxs, k=self.K)
            else:
                sel = random.sample(idxs, self.K)
            batch.extend(sel)
            if len(batch) >= self.P * self.K:
                yield from batch[:self.P * self.K]
                batch = batch[self.P * self.K:]
        if batch:
            yield from batch

    def __len__(self):
        return len(self.pids) * self.K


H, W = CFG["img_size"]
train_tf = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.Pad(10),
    transforms.RandomCrop((H, W)),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.4), value=0),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Fix: RandomErasing needs tensor input, reorder transforms
train_tf = transforms.Compose([
    transforms.Resize((H, W)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.Pad(10),
    transforms.RandomCrop((H, W)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.4), value=0),
])

train_ds = ReIDDataset(splits["train"], train_tf, is_train=True)
query_ds = ReIDDataset(splits["query"], val_tf, is_train=False)
gallery_ds = ReIDDataset(splits["gallery"], val_tf, is_train=False)

sampler = PKSampler(splits["train"], CFG["P"], CFG["K"])
train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], sampler=sampler,
                          num_workers=4, pin_memory=True, drop_last=True)
query_loader = DataLoader(query_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
gallery_loader = DataLoader(gallery_ds, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)} images, {CFG['num_classes']} IDs")
print(f"Query: {len(query_ds)}, Gallery: {len(gallery_ds)}")

In [ ]:
# ── TransReID ViT-B/16 Model ────────────────────────────────
import timm

class SideInformationEmbedding(nn.Module):
    """Camera-aware + view-aware side information embedding."""
    def __init__(self, num_cameras, embed_dim):
        super().__init__()
        self.camera_embed = nn.Embedding(num_cameras, embed_dim)
        nn.init.normal_(self.camera_embed.weight, std=0.02)

    def forward(self, camera_ids):
        return self.camera_embed(camera_ids)


class TransReIDViT(nn.Module):
    def __init__(self, num_classes, embed_dim=768, num_cameras=6, pretrained_path=None):
        super().__init__()
        # Load CLIP ViT-B/16 or timm ViT
        if pretrained_path and os.path.isfile(pretrained_path):
            self.vit = timm.create_model("vit_base_patch16_224", pretrained=False,
                                          img_size=CFG["img_size"], num_classes=0)
            state = torch.load(pretrained_path, map_location="cpu", weights_only=False)
            # CLIP state dict conversion
            if "state_dict" in state:
                state = state["state_dict"]
            elif "visual" in str(list(state.keys())[:5]):
                # OpenAI CLIP format
                visual_state = {}
                for k, v in state.items():
                    if k.startswith("visual."):
                        nk = k.replace("visual.", "")
                        # Map CLIP keys to timm keys
                        nk = nk.replace("transformer.resblocks", "blocks")
                        nk = nk.replace("ln_pre", "norm_pre")
                        nk = nk.replace("ln_post", "norm")
                        nk = nk.replace("ln_1", "norm1")
                        nk = nk.replace("ln_2", "norm2")
                        nk = nk.replace("in_proj_weight", "qkv.weight")
                        nk = nk.replace("in_proj_bias", "qkv.bias")
                        nk = nk.replace("out_proj", "proj")
                        nk = nk.replace("mlp.c_fc", "mlp.fc1")
                        nk = nk.replace("mlp.c_proj", "mlp.fc2")
                        nk = nk.replace("class_embedding", "cls_token")
                        nk = nk.replace("positional_embedding", "pos_embed")
                        nk = nk.replace("conv1", "patch_embed.proj")
                        visual_state[nk] = v
                # Handle cls_token shape
                if "cls_token" in visual_state:
                    visual_state["cls_token"] = visual_state["cls_token"].unsqueeze(0).unsqueeze(0)
                if "pos_embed" in visual_state:
                    visual_state["pos_embed"] = visual_state["pos_embed"].unsqueeze(0)
                msg = self.vit.load_state_dict(visual_state, strict=False)
            else:
                msg = self.vit.load_state_dict(state, strict=False)
            print(f"Loaded pretrained: missing={len(msg.missing_keys)}, unexpected={len(msg.unexpected_keys)}")
        else:
            self.vit = timm.create_model("vit_base_patch16_224", pretrained=True,
                                          img_size=CFG["img_size"], num_classes=0)
            print("Using ImageNet pretrained ViT")

        self.embed_dim = embed_dim
        self.sie = SideInformationEmbedding(num_cameras, embed_dim)

        # BNNeck
        self.bottleneck = nn.BatchNorm1d(embed_dim)
        self.bottleneck.bias.requires_grad_(False)

        # Classifier
        self.classifier = nn.Linear(embed_dim, num_classes, bias=False)
        nn.init.normal_(self.classifier.weight, std=0.001)

    def forward(self, x, camera_ids=None):
        # Get patch embeddings
        feat = self.vit.forward_features(x)

        # If we get a sequence, take CLS token
        if feat.dim() == 3:
            feat = feat[:, 0]

        # Add SIE
        if camera_ids is not None:
            feat = feat + self.sie(camera_ids)

        # BNNeck
        bn_feat = self.bottleneck(feat)

        if self.training:
            logits = self.classifier(bn_feat)
            return feat, bn_feat, logits
        return feat


model = TransReIDViT(
    num_classes=CFG["num_classes"],
    embed_dim=CFG["embed_dim"],
    num_cameras=CFG["num_cameras"],
    pretrained_path=PRETRAINED_PATH,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {n_params:,} params")

In [ ]:
# ── Losses ───────────────────────────────────────────────────
class LabelSmoothCE(nn.Module):
    def __init__(self, num_classes, smoothing=0.1):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing

    def forward(self, logits, targets):
        confidence = 1.0 - self.smoothing
        smooth_val = self.smoothing / self.num_classes
        one_hot = torch.full_like(logits, smooth_val)
        one_hot.scatter_(1, targets.unsqueeze(1), confidence + smooth_val)
        log_prob = F.log_softmax(logits, dim=1)
        loss = (-one_hot * log_prob).sum(dim=1).mean()
        return loss


class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, feats, labels):
        dist = torch.cdist(feats, feats, p=2)
        n = feats.size(0)
        mask_pos = labels.unsqueeze(0) == labels.unsqueeze(1)
        mask_neg = ~mask_pos

        # Hardest positive
        dist_pos = dist.clone()
        dist_pos[~mask_pos] = 0
        hardest_pos, _ = dist_pos.max(dim=1)

        # Hardest negative
        dist_neg = dist.clone()
        dist_neg[~mask_neg] = float('inf')
        hardest_neg, _ = dist_neg.min(dim=1)

        loss = F.relu(hardest_pos - hardest_neg + self.margin).mean()
        return loss


id_loss_fn = LabelSmoothCE(CFG["num_classes"], CFG["label_smooth"])
triplet_loss_fn = TripletLoss(CFG["triplet_margin"])

In [ ]:
# ── Optimizer & Scheduler ───────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"],
                               weight_decay=CFG["weight_decay"])

# Cosine with warmup
def lr_lambda(epoch):
    if epoch < CFG["warmup_epochs"]:
        return (epoch + 1) / CFG["warmup_epochs"]
    progress = (epoch - CFG["warmup_epochs"]) / max(1, CFG["epochs"] - CFG["warmup_epochs"])
    return 0.5 * (1 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=CFG["fp16"])
print(f"AdamW: lr={CFG['lr']}, wd={CFG['weight_decay']}, warmup={CFG['warmup_epochs']}ep")

In [ ]:
# ── Evaluation ───────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, query_loader, gallery_loader):
    model.eval()
    def extract(loader):
        feats, pids, cams = [], [], []
        for imgs, pid, cam in loader:
            imgs = imgs.to(device)
            f = model(imgs)
            f = F.normalize(f, p=2, dim=1)
            feats.append(f.cpu())
            pids.append(pid if isinstance(pid, torch.Tensor) else torch.tensor(pid))
            cams.append(cam if isinstance(cam, torch.Tensor) else torch.tensor(cam))
        return torch.cat(feats), torch.cat(pids), torch.cat(cams)

    qf, qp, qc = extract(query_loader)
    gf, gp, gc = extract(gallery_loader)

    # Cosine similarity (features are L2-normalized)
    dist = 1 - torch.mm(qf, gf.t())  # distance matrix

    # Compute CMC and mAP
    indices = dist.sort(dim=1)[1]
    all_cmc = []
    all_ap = []

    for qi in range(len(qp)):
        q_pid = qp[qi].item()
        q_cam = qc[qi].item()
        order = indices[qi]
        g_pids = gp[order]
        g_cams = gc[order]

        # Remove same camera same ID (standard Market eval protocol)
        keep = ~((g_pids == q_pid) & (g_cams == q_cam))
        g_pids = g_pids[keep]

        matches = (g_pids == q_pid).numpy().astype(np.int32)
        if matches.sum() == 0:
            continue

        # CMC
        cmc = matches.cumsum()
        cmc[cmc > 1] = 1
        all_cmc.append(cmc[:50])

        # AP
        n_rel = matches.sum()
        cum_match = matches.cumsum()
        precision = cum_match / (np.arange(len(matches)) + 1)
        ap = (precision * matches).sum() / n_rel
        all_ap.append(ap)

    all_cmc = np.array(all_cmc)
    cmc = all_cmc.mean(axis=0)
    mAP = np.mean(all_ap)

    return mAP, cmc[0], cmc[4], cmc[9]

In [ ]:
# ── Training ─────────────────────────────────────────────────
best_mAP = 0.0
history = []

print(f"Training {CFG['epochs']} epochs on Market-1501 ({CFG['num_classes']} IDs)")
print(f"Batch: P={CFG['P']} × K={CFG['K']} = {CFG['batch_size']}")
print("=" * 60)

for epoch in range(CFG["epochs"]):
    model.train()
    running_id, running_tri, n_batches = 0.0, 0.0, 0
    t0 = time.time()

    for imgs, labels, cam_ids in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        cam_ids = cam_ids.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=CFG["fp16"]):
            feat, bn_feat, logits = model(imgs, cam_ids)
            loss_id = id_loss_fn(logits, labels)
            loss_tri = triplet_loss_fn(feat, labels)
            loss = loss_id + loss_tri

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_id += loss_id.item()
        running_tri += loss_tri.item()
        n_batches += 1

    scheduler.step()
    elapsed = time.time() - t0

    if (epoch + 1) % 10 == 0 or epoch == 0:
        avg_id = running_id / max(n_batches, 1)
        avg_tri = running_tri / max(n_batches, 1)
        print(f"E{epoch+1:03d} ({elapsed:.0f}s) ID={avg_id:.3f} Tri={avg_tri:.3f} lr={scheduler.get_last_lr()[0]:.6f}")

    if (epoch + 1) % CFG["eval_every"] == 0 or epoch == CFG["epochs"] - 1:
        mAP, r1, r5, r10 = evaluate(model, query_loader, gallery_loader)
        print(f"  >> mAP={mAP:.4f}  R1={r1:.4f}  R5={r5:.4f}  R10={r10:.4f}")
        history.append({"epoch": epoch + 1, "mAP": float(mAP), "R1": float(r1), "R5": float(r5), "R10": float(r10)})
        if mAP > best_mAP:
            best_mAP = mAP
            torch.save({
                "model_state_dict": model.state_dict(),
                "epoch": epoch + 1,
                "mAP": mAP,
                "R1": r1,
                "config": CFG,
            }, "/kaggle/working/person_transreid_market1501_best.pth")
            print(f"  * New best! mAP={mAP:.4f} saved")

print(f"
Training complete. Best mAP={best_mAP:.4f}")

# Final eval
mAP, r1, r5, r10 = evaluate(model, query_loader, gallery_loader)
print(f"Final eval: mAP={mAP:.4f}  R1={r1:.4f}  R5={r5:.4f}  R10={r10:.4f}")

In [ ]:
# ── Export ────────────────────────────────────────────────────
out_path = Path("/kaggle/working/person_transreid_market1501_best.pth")
if out_path.exists():
    sz = out_path.stat().st_size / 1e6
    print(f"Exported: {out_path} ({sz:.1f} MB)")
else:
    print("WARNING: No best checkpoint saved!")

# Save training history
with open("/kaggle/working/training_history_person_reid.json", "w") as f:
    json.dump({"config": CFG, "history": history, "best_mAP": best_mAP}, f, indent=2)
    print(f"History: {len(history)} eval points")

print("
Done! Upload person_transreid_market1501_best.pth to mrkdagods/mtmc-weights/reid/")